In [1]:
import torch
import json
import io
import numpy as np
import scipy.special as sp

import pickle as pkl
import zlib
import base64

/Users/aleksei/.local/share/virtualenvs/kutulu-_n6nfavE/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from hydra import initialize, compose
from omegaconf import OmegaConf

In [3]:
import sys
sys.path.insert(0, '../')

In [4]:
from src.envs.agents.ppo_agent import PPOAgent

from src.envs.kutulu_observer import KutuluClosestObserver, KutuluClosestExtObserver
from src.envs.kutulu_world import KutuluWorldEnv
from src.game.template import (
    CELL_WALL, DEFAULT_KUTULU_ACTIONS, EXTENDED_KUTULU_ACTIONS, PPOConvSolver, PPOConvExtSolver, DQNConvSolver, DQNSolver,
)
from src.game.template import MOVE_REL_POS, REL_POSITIONS
from src.envs.agent_validator import AgentValidator
from src.game.template import DEFAULT_KUTULU_ACTIONS, EXTENDED_KUTULU_ACTIONS, UnreachedPositionError

In [5]:
# experiment = '0679b48e9b374e0f9c82535d01448c51'

In [6]:
# experiment = '05abe073de06428e896fcd880c9f3eac'

In [7]:
# experiment = '6d75a83e942740d2b077f6e815942a15'

In [8]:
# experiment = 'cd5d12509c5243f8b5fd8eb2fef0fdd9'

In [9]:
# 148 / 314
experiment = '9ad30a911ecb4619b530c09a000f80de'

In [10]:
# 5 / 314
experiment = 'c1ef7ae2d0e84408bee832b580b2b5d8'

In [11]:
# 23 / 314
experiment = 'd00fa77b3b3d4a4d9a0b1b7203b7de47'

In [12]:
# 4 / 314
experiment = '08d2f5ac186a44fb892f43181e199273'

In [40]:
mode = 'ppo_conv'

In [42]:
# 3 / 314
experiment = 'b522544f675143a6965c89d20500e446'
mode = 'ppo_conv_ext'

In [43]:
config_path = f'../../kutulu_artifacts/mlflow_artifacts/{experiment}/artifacts/hydra_config'
checkpoint_dir = f'../../kutulu_artifacts/mlflow_artifacts/{experiment}/artifacts/models/agent_0/final'

In [44]:
with initialize(version_base=None, config_path=config_path):
    cfg = compose(config_name="config")

In [45]:
info = OmegaConf.to_container(cfg.agent, resolve=True)
del info['type']

In [46]:
agent = PPOAgent(**info)

In [47]:
agent.model.load_state_dict(torch.load(f"{checkpoint_dir}/model.pt"))

<All keys matched successfully>

In [48]:
agent.train = True

In [49]:
av = AgentValidator(EXTENDED_KUTULU_ACTIONS)
av_plan = AgentValidator(EXTENDED_KUTULU_ACTIONS, player_params=(100, 1, 0))

In [50]:
av.check_entity_nearby(agent, 'EXPLORER', n_min=2, n_max=3)

(1.0, 0.0, 4, 0.0, 0.0)

In [51]:
av_plan.check_entity_nearby(agent, 'EXPLORER', n_min=1, n_max=2)

(0.625, 0.0, 1, 0.0, 0.0)

In [52]:
av_plan.check_entity_nearby(agent, 'EXPLORER', n_min=3, n_max=3)

(1.0, 0.0, 4, 0.0, 0.0)

In [53]:
av.check_entity_nearby(agent, 'WANDERER', n_min=1, n_max=2)

(1.0, 0.0, 1, 0.0, 0.0)

In [54]:
av.check_entity_nearby(agent, 'WANDERER', n_min=1, n_max=2, env_types=['corner'])

(1.0, 0.0, 4, 0.0, 0.0)

In [55]:
av.check_entity_nearby(agent, 'WANDERER', n_min=1, n_max=2, env_types=['coridor'])

(1.0, 0.0, 4, 0.0, 0.0)

In [56]:
agent.train = False

In [57]:
weights = {}
for k,v in agent.model.state_dict().items():
    weights[k] = v.detach().numpy()
    print(k, v.shape)

conv1.weight torch.Size([8, 18, 3, 3])
conv1.bias torch.Size([8])
bn1.weight torch.Size([8])
bn1.bias torch.Size([8])
bn1.running_mean torch.Size([8])
bn1.running_var torch.Size([8])
bn1.num_batches_tracked torch.Size([])
conv2.weight torch.Size([8, 8, 3, 3])
conv2.bias torch.Size([8])
bn2.weight torch.Size([8])
bn2.bias torch.Size([8])
bn2.running_mean torch.Size([8])
bn2.running_var torch.Size([8])
bn2.num_batches_tracked torch.Size([])
fc.weight torch.Size([16, 72])
fc.bias torch.Size([16])
actor.weight torch.Size([8, 16])
actor.bias torch.Size([8])
critic.weight torch.Size([1, 16])
critic.bias torch.Size([1])
terminator.weight torch.Size([1, 16])
terminator.bias torch.Size([1])


In [58]:
env = KutuluWorldEnv('', '', 1, actions=EXTENDED_KUTULU_ACTIONS)
env.map = [
    '###########',
    '#.........#',
    '#.#.#.#.#.#',
    '#.........#',
    '#.#.#.#.#.#',
    '#.........#',
    '###########',
]
env.width = len(env.map[0])
env.height = len(env.map)

In [59]:
from tests.utils import calculate_entities

In [60]:
player_pos = (3, 3)
explorers = [(5, 3), (1, 3)]
wanderers = [(3, 1, 1), (3, 5, 1), (3, 2, 0)]

entities = calculate_entities(player_pos, explorers, wanderers)
agent.set_env(env)
env._set_entities(entities)
env._set_players(entities, set_ids=True)

state = agent.observer.get_state(0)
# test_data = agent.episode_buffer.encode_states([state], return_tensors=False)
tensor_data = agent.episode_buffer.state_encoder.encode_states([state], return_tensors=True)

In [62]:
info = {
    'width': env.width,
    'height': env.height,
    'lines': env.map,
}

# solver = PPOConvExtSolver(info, EXTENDED_KUTULU_ACTIONS, weights, size=agent.size)

In [66]:
USED_ACTIONS = EXTENDED_KUTULU_ACTIONS
checkpoint_data = weights
SIZE = agent.size

In [72]:
if mode == 'qlearning':
    solver = QlearningSolver(info, USED_ACTIONS, checkpoint_data)
elif mode == 'dqn_ext':
    solver = DQNSolver(info, USED_ACTIONS, checkpoint_data)
elif mode == 'dqn_by_kind':
    solver = DQNByKindSolver(info, USED_ACTIONS, checkpoint_data)
elif mode == 'dqn_conv':
    solver = DQNConvSolver(info, USED_ACTIONS, checkpoint_data, SIZE)
elif mode == 'ppo_conv':
    solver = PPOConvSolver(info, USED_ACTIONS, checkpoint_data, SIZE)
elif mode == 'ppo_conv_ext':
    solver = PPOConvExtSolver(info, USED_ACTIONS, checkpoint_data, SIZE)
else:
    raise ValueError(f'unknown mode: "{mode}"')

In [73]:
np_output = solver._calculate_output([e.to_dict() for e in env._get_entites(0)], player_pos)

In [74]:
np_output

array([0.062851  , 0.17612319, 0.11435528, 0.19382111, 0.04445026,
       0.14224991, 0.12936374, 0.13678552])

In [75]:
model_output = agent.model(tensor_data)['policy'].detach().cpu().numpy()

In [76]:
model_output

array([[0.06285099, 0.17612317, 0.11435528, 0.19382113, 0.04445026,
        0.1422499 , 0.12936373, 0.13678552]], dtype=float32)

In [77]:
# data2, data1 = zip(*weights.items())

# data1 = pkl.dumps(data1)
# data2 = pkl.dumps(data2)

In [78]:
data1 = []
data2 = []
for k, v in weights.items():
    if 'num_batches_tracked' in k:
        continue
    print(k, v.shape)
    data2.append(k)
    buffer = io.BytesIO()
    v = v.astype(np.float16)
    np.save(buffer, v)
    data1.append(buffer.getvalue())
    # data1.append(zlib.compress(buffer.getvalue(), level=9))

data1 = pkl.dumps(data1)
data2 = pkl.dumps(data2)

conv1.weight (8, 18, 3, 3)
conv1.bias (8,)
bn1.weight (8,)
bn1.bias (8,)
bn1.running_mean (8,)
bn1.running_var (8,)
conv2.weight (8, 8, 3, 3)
conv2.bias (8,)
bn2.weight (8,)
bn2.bias (8,)
bn2.running_mean (8,)
bn2.running_var (8,)
fc.weight (16, 72)
fc.bias (16,)
actor.weight (8, 16)
actor.bias (8,)
critic.weight (1, 16)
critic.bias (1,)
terminator.weight (1, 16)
terminator.bias (1,)


In [79]:
with open('../src/game/template.py') as f:
    lines = f.readlines()

In [82]:
mode

'ppo_conv_ext'

In [83]:
with open('../src/game/template_submit.py', 'w') as f:
    for line in lines:
        line = line.replace("b'data1data1data1'", str(base64.b64encode(zlib.compress(data1, level=9))))
        line = line.replace("b'data2data2data2'", str(base64.b64encode(zlib.compress(data2, level=9))))
        line = line.replace("mode = 'mode'", f"mode = '{mode}'")
        line = line.replace("USED_ACTIONS = DEFAULT_KUTULU_ACTIONS", "USED_ACTIONS = EXTENDED_KUTULU_ACTIONS")
        line = line.replace("SIZE = 3", f"SIZE = {agent.size}")
        f.write(line)

In [84]:
!ls -lh ../src/game/template_submit.py

-rw-r--r--  1 aleksei  staff    51K 31 авг 21:28 ../src/game/template_submit.py
